# Information Horizon scores for 2 MELD dev samples

Run the notebook from top to bottom. It prepares the dev index, scores **every projected visual token at every decoder depth for the first 2 available dev samples**, summarizes the scores, and saves a download bundle. No experiment switches or pilot steps are required.

The score is `P(correct emotion | token k is the only visual token retained at depth d) − P(correct emotion | no visual tokens retained at depth d)`. The probabilities come from the saved full-baseline emotion MLP. This is an emotion-class adaptation of the paper's first-answer-token LM readout. The MLP and Qwen stay frozen.

Attach the current `benchmark-code` and `meld-dataset` Kaggle inputs. Enable a GPU and Internet. The code input must include the multi-GPU mask fix (`effective_mask.to(hidden.device)`) introduced after the first pilot failed on two T4s.

**Compute:** This exact intervention performs `1 + number_of_depths × (number_of_visual_tokens + 1)` forward passes **per sample**. Scoring 2 samples may still exceed a Kaggle session. Each completed sample is saved immediately. The notebook writes only the ground-truth information score per token to keep the output smaller; full and no-visual seven-class probabilities remain available per sample. Rerun the notebook with the same inputs to continue; do not delete its `/kaggle/working` output. Between separate Kaggle sessions, attach or restore the previous output directory before rerunning.

[Paper, Section 3.2](https://arxiv.org/html/2512.07580v2) · [Method details in this repository](https://github.com/YahongWang1/Information-Horizon)


## 1. Inputs and output


In [ ]:
from pathlib import Path

CODE = Path("/kaggle/input/datasets/pushkarsingh2005/benchmark-code")
MELD_ROOT = Path("/kaggle/input/datasets/pushkarsingh2005/meld-dataset/MELD.Raw")
CHECKPOINT = CODE / "experiments/baselines/qwen2_5_vl_3b_shared/outputs/best_mlp.pt"
RUN_ROOT = Path("/kaggle/working/experiments/information_horizon/qwen2_5_vl_3b_meld_dev_first2_mlp_all_tokens")
INDEX_DIR = RUN_ROOT / "data"
INDEX_CSV = INDEX_DIR / "meld_dev_index.csv"
SCORES_DIR = RUN_ROOT / "scores"
REPORT_DIR = RUN_ROOT / "reports"
RUNNER = CODE / "scripts/information_horizon/run_information_horizon.py"
SUMMARIZER = CODE / "scripts/information_horizon/summarize_information_horizon.py"

for required in (CODE, MELD_ROOT, CHECKPOINT, RUNNER, SUMMARIZER):
    assert required.exists(), f"Missing {required}. Attach the updated code and MELD datasets."
assert "effective_mask.to(hidden.device)" in RUNNER.read_text(), "Update the benchmark-code Kaggle dataset: this copy lacks the two-GPU mask fix."
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print("Scores:", SCORES_DIR)


## 2. Install dependencies

Keep Kaggle's CUDA-enabled PyTorch. This pins the Transformers version used to test the decoder hooks. Restart the kernel if a different version was already imported.


In [ ]:
!pip install -q "transformers==4.57.1" "qwen-vl-utils[decord]==0.0.14" accelerate decord pandas numpy scikit-learn matplotlib Pillow tqdm PyYAML opencv-python-headless


In [ ]:
import json
import os
import shutil
import numpy as np
import pandas as pd
import torch
import transformers
import matplotlib.pyplot as plt
from IPython.display import display, FileLink

assert transformers.__version__ == "4.57.1", "Restart the kernel after installing Transformers 4.57.1."
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator."
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name}, {props.total_memory / 2**30:.1f} GiB")
checkpoint = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
assert checkpoint["input_dim"] == 2048
assert checkpoint["label_names"] == ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]
del checkpoint


## 3. Prepare the dev split

The index is rebuilt on Kaggle so its absolute video paths point to the attached MELD dataset. `prepare_meld.py` includes available video files.


In [ ]:
!python {CODE}/scripts/data/prepare_meld.py \
  --meld-root {MELD_ROOT} \
  --split dev \
  --output-dir {INDEX_DIR}

index = pd.read_csv(INDEX_CSV)
assert len(index) > 0 and not index.sample_id.duplicated().any()
assert index.video_path.map(lambda path: Path(path).is_file()).all()
SAMPLE_COUNT = min(2, len(index))
print("Dev samples to score:", SAMPLE_COUNT, "of", len(index))
display(index[["sample_id", "emotion", "utterance"]].head(SAMPLE_COUNT))


## 4. Score all samples, tokens, and depths

This cell starts scoring the first 2 available dev samples immediately. One JSON file is written per completed sample. `--resume` skips completed samples on rerun and checks that the configuration and index are unchanged. The scoring process may need more than one Kaggle session; the next session requires the previous `RUN_ROOT` restored to the same path.


In [ ]:
!python {RUNNER} \
  --index-csv {INDEX_CSV} \
  --output-dir {SCORES_DIR} \
  --checkpoint {CHECKPOINT} \
  --readout mlp \
  --depths all \
  --max-tokens 0 \
  --score-only \
  --limit {SAMPLE_COUNT} \
  --start 0 \
  --fps 6 \
  --min-frames 4 \
  --max-frames 64 \
  --frame-size 224 \
  --pooling last \
  --prompt-style emotion_task \
  --dtype float16 \
  --resume


## 5. Verify completion and summarize

The notebook expects one result file for every indexed sample. If the scoring cell stopped early, rerun it after restoring the saved output. This check prevents a partial run from looking like a full result.


In [ ]:
sample_files = sorted(path for path in SCORES_DIR.glob("*.json") if path.name != "config.json")
assert len(sample_files) == SAMPLE_COUNT, f"Only {len(sample_files)}/{SAMPLE_COUNT} samples completed. Rerun the scoring cell with the same settings."
completed_ids = set()
for path in sample_files:
    record = json.loads(path.read_text())
    assert len(record["scored_token_ordinals"]) == record["visual_token_count"]
    assert all(len(layer["tokens"]) == record["visual_token_count"] for layer in record["depths"])
    completed_ids.add(record["sample_id"])
assert completed_ids == set(index.head(SAMPLE_COUNT).sample_id), "Completed sample IDs differ from the selected dev samples."
print("Complete:", len(sample_files), "samples; all projected visual tokens scored at all configured depths")
!python {SUMMARIZER} --input-dir {SCORES_DIR}
summary = pd.read_csv(SCORES_DIR / "layer_summary.csv")
display(summary)


## 6. Export scores and class probabilities

`token_information_scores.csv` has one row per sample, decoder depth, and visual token: the signed score and its ground-truth present probability. `full_vs_absent_probabilities.csv` retains all seven class probabilities for the full and no-visual branches.

Rows are written one sample at a time so the export does not hold the entire experiment in memory.


In [ ]:
import csv

REPORT_DIR.mkdir(parents=True, exist_ok=True)
labels = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]
token_fields = ["sample_id", "true_label", "depth", "token_ordinal", "sequence_position",
                "information_score", "present_true_probability", "absent_true_probability"]
probability_fields = ["sample_id", "true_label", "depth", "emotion", "full_probability",
                      "absent_probability", "full_minus_absent"]
row_count = 0
with (REPORT_DIR / "token_information_scores.csv").open("w", newline="") as token_file, \
     (REPORT_DIR / "full_vs_absent_probabilities.csv").open("w", newline="") as probability_file:
    token_writer = csv.DictWriter(token_file, fieldnames=token_fields)
    probability_writer = csv.DictWriter(probability_file, fieldnames=probability_fields)
    token_writer.writeheader()
    probability_writer.writeheader()
    for path in sample_files:
        sample = json.loads(path.read_text())
        assert sample["labels"] == labels and sample["readout"] == "mlp"
        true_id = labels.index(sample["true_label"])
        for layer in sample["depths"]:
            for j, emotion in enumerate(labels):
                probability_writer.writerow({
                    "sample_id": sample["sample_id"], "true_label": sample["true_label"],
                    "depth": layer["depth"], "emotion": emotion,
                    "full_probability": sample["full_probabilities"][j],
                    "absent_probability": layer["absent_probabilities"][j],
                    "full_minus_absent": layer["full_minus_absent"][j],
                })
            for token in layer["tokens"]:
                token_writer.writerow({
                    "sample_id": sample["sample_id"], "true_label": sample["true_label"],
                    "depth": layer["depth"], "token_ordinal": token["token_ordinal"],
                    "sequence_position": token["sequence_position"],
                    "information_score": token["information_score"],
                    "present_true_probability": token["present_true_probability"],
                    "absent_true_probability": layer["absent_probabilities"][true_id],
                })
                row_count += 1
summary.to_csv(REPORT_DIR / "layer_summary.csv", index=False)
print("Exported", row_count, "token-depth scores to", REPORT_DIR)
display(pd.read_csv(REPORT_DIR / "token_information_scores.csv", nrows=5))


## 7. Plot the information profile

Inspect mean absolute information and within-sample variance alongside the signed mean. A signed mean near zero can hide positive and negative token effects cancelling. A later text token may already contain earlier visual evidence.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
axes[0, 0].plot(summary.depth, summary.mean_information, "o-", label="Mean signed I")
axes[0, 0].plot(summary.depth, summary.mean_absolute_information, "o-", label="Mean |I|")
axes[0, 0].axhline(0, color="gray", lw=.7)
axes[0, 0].legend()
axes[0, 0].set_ylabel("Ground-truth probability difference")
axes[0, 1].plot(summary.depth, summary.within_sample_variance, "o-")
axes[0, 1].set_ylabel("Mean within-sample variance")
axes[1, 0].plot(summary.depth, summary.full_correct, "o-", label="Full tokens")
axes[1, 0].plot(summary.depth, summary.absent_correct, "o-", label="No visual tokens after depth")
axes[1, 0].set_ylabel("Accuracy across selected dev samples")
axes[1, 0].set_ylim(-.03, 1.03)
axes[1, 0].legend()
axes[1, 1].plot(summary.depth, summary.full_minus_absent_true, "o-")
axes[1, 1].axhline(0, color="gray", lw=.7)
axes[1, 1].set_ylabel("Mean P_full(y) - P_absent(y)")
for ax in axes.flat:
    ax.set_xlabel("Completed decoder blocks")
    ax.grid(alpha=.2)
fig.suptitle(f"Information Horizon: MELD dev, {SAMPLE_COUNT} samples, all visual tokens")
fig.savefig(REPORT_DIR / "information_horizon.png", dpi=160)
plt.show()


## 8. Save a download bundle

The bundle contains one JSON per sample, CSV summaries, plots, and the index. It excludes video files, model weights, and the MLP checkpoint. Kaggle notebook versions also preserve `/kaggle/working` outputs. The visual-token index is an ordinal in the processed Qwen sequence, not a pixel-coordinate face map.


In [ ]:
archive = shutil.make_archive(str(RUN_ROOT), "zip", root_dir=RUN_ROOT.parent, base_dir=RUN_ROOT.name)
print("Download bundle:", archive)
display(FileLink(os.path.relpath(archive, Path.cwd())))
